# 📖 Notebook 3: Backup Strategies

## Why This Matters

Replication protects you from hardware failure, but NOT from:
- **Accidental DELETE** — someone runs `DELETE FROM orders` without a WHERE clause
- **Ransomware** — malicious encryption of your data (replicated to standby!)
- **Logical corruption** — a bug writes bad data to the database
- **Compliance** — regulations require you to keep historical backups

**Replication is NOT a backup.** If you delete data on the primary,
the delete is replicated to the standby. You need actual backups.

## Learning Objectives

- Understand full, incremental, and differential backup types
- Perform a logical backup with `pg_dump`
- Perform a physical backup with `pg_basebackup`
- Test backup restoration (the most neglected practice!)
- Understand point-in-time recovery (PITR) concepts

In [ ]:
# ── Preflight: FAIL BACK from notebook 2's failover ──────────────────────────
#
# Notebook 2 stops mid-disaster on purpose: the primary is FENCED (its
# container is stopped -- the STONITH step of a real failover) and the standby
# is PROMOTED. That is the state an on-call engineer is actually handed, and
# it is not a state this notebook can run in:
#
#   * port 5432 is dead, so every connection below would fail, and
#   * the promoted node on 55433 is now an INDEPENDENT primary holding writes
#     the fenced node never saw. Simply restarting the fenced node gives you
#     two primaries with diverged data -- textbook split-brain -- and throws
#     the promoted node's writes away without saying so.
#
# So this cell performs a real FAILBACK, in runbook order:
#   1. capture the rows written on the promoted node during the failover
#      window (that divergence is the whole problem),
#   2. un-fence the old primary,
#   3. re-apply the captured rows, so nothing written during the outage is
#      lost -- a failback that drops them is data loss with a success message,
#   4. re-clone the standby from the primary so streaming replication resumes,
#   5. verify: primary writable, standby in recovery AND streaming, and the
#      carried-back rows visible on both.
#
# Production does step 3 with `pg_rewind` plus WAL replay, which preserves
# every table and every row id. We do it at row level because this lab's
# failover-window writes all land in one table (`audit_log`) -- the lesson is
# the same. Calling this when nothing is wrong is a no-op, so it is safe to
# re-run and safe if you never ran notebook 2 at all.
import socket
import subprocess
import time
from pathlib import Path

import psycopg2

PRIMARY_PORT, STANDBY_PORT = 5432, 55433
PRIMARY_SERVICE, STANDBY_SERVICE = "pg-primary", "pg-standby"
PRIMARY_CONTAINER, STANDBY_CONTAINER = "bcdr-pg-primary", "bcdr-pg-standby"
PGDATA_PATH = "/var/lib/postgresql/data"
AUDIT_COLS = "table_name, record_id, action, changed_by, changed_at"


def _lab_root():
    """The directory holding docker-compose.yml, found from wherever we are."""
    here = Path.cwd().resolve()
    for cand in [here, *here.parents]:
        if (cand / "docker-compose.yml").exists():
            return cand
    raise RuntimeError(f"no docker-compose.yml at or above {here}")


LAB_ROOT = _lab_root()


def _run(cmd, cwd=None, timeout=300):
    return subprocess.run(cmd, cwd=cwd and str(cwd),
                          capture_output=True, text=True, timeout=timeout)


def port_open(port, host="127.0.0.1", timeout=1.0):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as sock:
        sock.settimeout(timeout)
        return sock.connect_ex((host, port)) == 0


def pg_query(port, sql, params=None):
    """Rows from the node on `port`, or None if it is not answering.

    READ-ONLY on purpose: the connection is closed without committing, so
    psycopg2 rolls back. Use it to inspect a node, never to change one --
    `carry_back()` below opens its own autocommit connection for writes."""
    try:
        conn = psycopg2.connect(host="127.0.0.1", port=port, dbname="bcdr_demo",
                                user="demo", password="demo", connect_timeout=3)
    except psycopg2.Error:
        return None
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            return cur.fetchall()
    except psycopg2.Error:
        return None
    finally:
        conn.close()


def node_role(port):
    """'down', 'primary' (writable) or 'standby' (in recovery)."""
    rows = pg_query(port, "SELECT pg_is_in_recovery()")
    if rows is None:
        return "down"
    return "standby" if rows[0][0] else "primary"


def streaming_ok():
    rows = pg_query(PRIMARY_PORT,
                    "SELECT 1 FROM pg_stat_replication WHERE state = 'streaming'")
    return bool(rows)


def topology_ok():
    """The documented shape: writable primary on 5432, streaming standby on 55433."""
    return (node_role(PRIMARY_PORT) == "primary"
            and node_role(STANDBY_PORT) == "standby"
            and streaming_ok())


def wait_topology(timeout=120):
    """Give a cold `docker compose up` time to finish the standby's basebackup
    before we conclude anything is wrong."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        if topology_ok():
            return True
        time.sleep(2)
    return topology_ok()


def audit_rows(port):
    rows = pg_query(port, f"SELECT {AUDIT_COLS} FROM audit_log")
    return None if rows is None else {tuple(r) for r in rows}


def start_primary(timeout=180):
    """Un-fence the old primary and wait until it is writable again."""
    _run(["docker", "compose", "up", "-d", PRIMARY_SERVICE], cwd=LAB_ROOT)
    deadline = time.time() + timeout
    while time.time() < deadline:
        if node_role(PRIMARY_PORT) == "primary":
            return
        time.sleep(2)
    raise RuntimeError(
        "the fenced primary did not come back on port 5432. Reset the lab:\n"
        "    docker compose down -v && docker compose up -d --wait")


def carry_back(rows):
    """Re-apply failover-window rows onto the primary, keeping their original
    timestamps. Row ids are NOT preserved -- they came from the promoted
    node's own sequence. `pg_rewind` would keep them; row-level carry-back
    cannot, which is one honest reason production prefers pg_rewind."""
    if not rows:
        return 0
    conn = psycopg2.connect(host="127.0.0.1", port=PRIMARY_PORT,
                            dbname="bcdr_demo", user="demo", password="demo")
    conn.autocommit = True
    try:
        with conn.cursor() as cur:
            for r in sorted(rows, key=lambda row: row[4]):
                cur.execute(
                    "INSERT INTO audit_log (table_name, record_id, action, "
                    "changed_by, changed_at) VALUES (%s, %s, %s, %s, %s)", r)
    finally:
        conn.close()
    return len(rows)


def standby_volume():
    """The named volume backing the standby's PGDATA (asked, not guessed)."""
    fmt = ('{{range .Mounts}}{{if eq .Destination "' + PGDATA_PATH
           + '"}}{{.Name}}{{end}}{{end}}')
    name = _run(["docker", "inspect", "-f", fmt, STANDBY_CONTAINER]).stdout.strip()
    return name or f"{LAB_ROOT.name}_pg_standby_data"


def rebuild_standby(timeout=300):
    """Re-clone the standby from the primary. A promoted node has switched to
    a new WAL timeline and cannot just reattach; real clusters run pg_rewind
    here, but a lab-sized database is faster to clone from scratch."""
    vol = standby_volume()
    _run(["docker", "compose", "rm", "-sf", STANDBY_SERVICE], cwd=LAB_ROOT)
    _run(["docker", "volume", "rm", "-f", vol])
    # pg_basebackup reuses 'standby_slot' and refuses while the slot still
    # shows a live walsender from the node we just removed.
    deadline = time.time() + 60
    while time.time() < deadline:
        rows = pg_query(PRIMARY_PORT,
                        "SELECT active FROM pg_replication_slots "
                        "WHERE slot_name = 'standby_slot'")
        if not rows or not rows[0][0]:
            break
        time.sleep(1)
    _run(["docker", "compose", "up", "-d", STANDBY_SERVICE], cwd=LAB_ROOT)
    deadline = time.time() + timeout
    while time.time() < deadline:
        if node_role(STANDBY_PORT) == "standby" and streaming_ok():
            return
        time.sleep(2)
    raise RuntimeError(
        "the standby did not come back as a streaming standby. Reset the lab:\n"
        "    docker compose down -v && docker compose up -d --wait")


def failback():
    """Restore the documented topology WITHOUT losing failover-window writes."""
    broken_now = (node_role(PRIMARY_PORT) != "primary"
                  or node_role(STANDBY_PORT) == "primary")
    if not broken_now and wait_topology():
        print("✅ Topology is already the documented one: "
              "primary on 5432, streaming standby on 55433. Nothing to do.")
        return {"action": "none", "carried_back": 0, "carried_markers": []}

    print("⚠️  Topology is not the documented one — running FAILBACK.")
    print(f"    port 5432: {node_role(PRIMARY_PORT):<8} "
          f"port 55433: {node_role(STANDBY_PORT)}")

    # 1. Capture the divergence while the promoted node is still its only home.
    promoted = (audit_rows(STANDBY_PORT)
                if node_role(STANDBY_PORT) == "primary" else None)

    # 2. Un-fence the old primary.
    start_primary()
    print("    ✅ old primary is back on 5432")

    # 3. Carry the failover-window writes forward.
    carried, markers = 0, []
    if promoted is None:
        print("    ·  the promoted node was not reachable — nothing to carry back")
    else:
        divergent = promoted - (audit_rows(PRIMARY_PORT) or set())
        markers = sorted({r[3] for r in divergent if r[3]})
        carried = carry_back(divergent)
        print(f"    ✅ carried {carried} row(s) written during the failover "
              f"window back onto the primary")
        if markers:
            print(f"       markers: {', '.join(markers)}")

    # 4. Re-form the pair.
    #
    #    A standby that was merely stopped or briefly detached -- which is
    #    where a failover that dies half-way leaves you, with no writable node
    #    anywhere -- can reattach on its own now that the primary is back.
    #    Wiping a perfectly good standby to prove a point is a second outage,
    #    so try that first.
    #
    #    A PROMOTED node is different in kind: it switched to its own WAL
    #    timeline the moment it was promoted, so it can never reattach to the
    #    old primary. That one has to be re-cloned (production: pg_rewind).
    reattached = False
    if promoted is None:
        _run(["docker", "compose", "up", "-d", STANDBY_SERVICE], cwd=LAB_ROOT)
        reattached = wait_topology(timeout=60)
    if reattached:
        print("    ✅ standby reattached to the restarted primary — no re-clone needed")
    else:
        print("    …  re-cloning the standby from the primary (takes a moment)")
        rebuild_standby()
        print("    ✅ standby is streaming again")

    # 5. Verify. A failback you did not verify is a failback you did not do.
    assert node_role(PRIMARY_PORT) == "primary", "5432 is not a writable primary"
    assert node_role(STANDBY_PORT) == "standby", "55433 is not in recovery"
    assert streaming_ok(), "no standby is streaming from the primary"
    if markers:
        deadline, seen = time.time() + 30, 0
        while time.time() < deadline:
            rows = pg_query(
                STANDBY_PORT,
                "SELECT COUNT(*) FROM audit_log WHERE changed_by = ANY(%s)",
                (markers,))
            seen = rows[0][0] if rows else 0
            if seen >= len(markers):
                break
            time.sleep(1)
        assert seen >= len(markers), (
            f"only {seen}/{len(markers)} failover-window write(s) reached the "
            f"rebuilt standby -- the failback lost data it claimed to preserve")
        print(f"    ✅ all {len(markers)} failover-window write(s) survived the "
              f"failback and replicated to the new standby")

    print("✅ FAILBACK complete: primary on 5432, streaming standby on 55433.")
    return {"action": "failback", "carried_back": carried,
            "carried_markers": markers}


failback()


## 🛠️ Setup

The preflight cell above already put the cluster back into the documented
shape (writable primary on 5432, streaming standby on 55433), whether you
arrived here straight from a cold start or from notebook 2's failover.

You only need the big hammer if the preflight tells you it could not:

```bash
cd 08-enterprise/bcdr
docker compose down -v && docker compose up -d --wait
```

Note what that command costs you: `-v` deletes the volumes, so every write
made in notebooks 1 and 2 — including anything written on the promoted node
during the failover — is gone. That is fine in a lab and unthinkable in
production, which is why the preflight does a real failback instead.

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).


In [ ]:
import psycopg2
import subprocess
import time
import os
from tabulate import tabulate

DB_PRIMARY = {
    "host": "localhost", "port": 5432,
    "database": "bcdr_demo", "user": "demo", "password": "demo"
}

def get_primary_connection():
    return psycopg2.connect(**DB_PRIMARY)

def docker_exec(container, cmd):
    result = subprocess.run(
        ["docker", "exec", container] + cmd,
        capture_output=True, text=True, timeout=60
    )
    return result.stdout.strip(), result.stderr.strip()

# Test connection
conn = get_primary_connection()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM orders")
print(f"✅ Connected. Orders in database: {cur.fetchone()[0]}")
conn.close()

## 📚 Backup Types Explained

### Full Backup
- Copies **everything** in the database
- Slowest to create, largest file size
- Fastest to restore (just load one file)
- **Example**: `pg_dump` or `pg_basebackup`

### Incremental Backup
- Copies only data that changed **since the last backup of any type**
- Fastest to create, smallest file size
- Slowest to restore (need full + every incremental in order)
- **Example**: WAL archiving between `pg_basebackup` runs

### Differential Backup
- Copies only data that changed **since the last full backup**
- Medium speed to create, medium file size
- Medium restore speed (need full + latest differential)

```
         Day 1       Day 2       Day 3       Day 4       Day 5
Full:   [AAABBB]
Incr:               [CC]        [DD]        [EE]        [FF]
Diff:               [CC]        [CCDD]      [CCDDEE]    [CCDDEEFF]

To restore Day 5:
  Full only:  Not possible (Day 1 data only)
  Incremental: Full + CC + DD + EE + FF (5 files)
  Differential: Full + CCDDEEFF (2 files)
```

In [ ]:
# =============================================================================
# Demo: Logical Backup with pg_dump
# =============================================================================
# pg_dump creates a SQL script that recreates your database.
# It is a LOGICAL backup — human-readable SQL statements.

print("=" * 65)
print("LOGICAL BACKUP WITH pg_dump")
print("=" * 65)

# Create a full logical backup
start_time = time.time()
out, err = docker_exec('bcdr-pg-primary', [
    "pg_dump",
    "-U", "demo",
    "-d", "bcdr_demo",
    "--format=custom",       # compressed binary format
    "--file=/tmp/backup.dump"
])
backup_time = time.time() - start_time

assert 'error' not in err.lower(), (
    f"pg_dump failed, so there is no backup to verify: {err}")
print(f"  ✅ Backup completed in {backup_time:.2f} seconds")

# Check backup size
out, _ = docker_exec('bcdr-pg-primary', ["ls", "-lh", "/tmp/backup.dump"])
print(f"  📦 Backup file: {out}")

# Verify backup contents (list what is inside)
out, _ = docker_exec('bcdr-pg-primary', ["pg_restore", "--list", "/tmp/backup.dump"])
# `pg_restore --list` opens with a ';'-commented header block. Those lines are
# not objects; counting them would overstate what is actually in the backup.
objects = [l for l in out.split('\n')
           if l.strip() and not l.lstrip().startswith(';')]
print(f"  📋 Backup contains {len(objects)} objects")
print("\n  First 10 objects in backup:")
for line in objects[:10]:
    print(f"    {line}")

# The five seeded tables must all be in there. A backup that silently captured
# nothing is the GitLab failure mode, and it looks exactly like a success.
listing = "\n".join(objects)
for tbl in ('customers', 'orders', 'order_items', 'payments', 'audit_log'):
    assert f" {tbl} " in listing, f"table '{tbl}' is missing from the backup"
print(f"\n  ✅ All five seeded tables are present in the archive listing.")


## 📚 Physical Backup with pg_basebackup

`pg_basebackup` creates a **physical copy** of the entire database cluster.
This is the same tool we used to set up our standby server!

### Logical vs Physical Backup

| Feature | Logical (pg_dump) | Physical (pg_basebackup) |
|---------|------------------|-------------------------|
| What it copies | Table data + schema | Entire data directory (binary) |
| Speed | Slower (reads every row) | Faster (copies files) |
| Selective restore | Yes (individual tables) | No (all or nothing) |
| Cross-version | Yes (can restore to newer PG) | No (same major version only) |
| Point-in-time recovery | No | Yes (with WAL archiving) |
| Best for | Small databases, migrations | Large databases, PITR |

In [ ]:
# =============================================================================
# Demo: Physical Backup with pg_basebackup
# =============================================================================

print("=" * 65)
print("PHYSICAL BACKUP WITH pg_basebackup")
print("=" * 65)

# pg_basebackup refuses to write into a non-empty directory, so clear the
# target first -- otherwise re-running this cell fails with a confusing error.
docker_exec('bcdr-pg-primary', ["rm", "-rf", "/tmp/physical_backup"])

start_time = time.time()
out, err = docker_exec('bcdr-pg-primary', [
    "pg_basebackup",
    "-U", "demo",
    "-D", "/tmp/physical_backup",
    "-Ft",    # tar format
    "-z",     # gzip compression
    "-Xs",    # stream WAL during backup
    "-P"      # show progress
])
backup_time = time.time() - start_time

print(f"  Time: {backup_time:.2f} seconds")
if err:
    # pg_basebackup outputs progress to stderr
    progress_lines = [l for l in err.split('\n') if '%' in l]
    if progress_lines:
        print(f"  Progress: {progress_lines[-1].strip()}")

# Check backup files
out, _ = docker_exec('bcdr-pg-primary', ["ls", "-lh", "/tmp/physical_backup/"])
print(f"\n  Backup files:")
for line in out.split('\n'):
    if line.strip():
        print(f"    {line.strip()}")

names, _ = docker_exec('bcdr-pg-primary', ["ls", "-1", "/tmp/physical_backup/"])
files = [l.strip() for l in names.split('\n') if l.strip()]

print("\n💡 base.tar.gz = all database files, pg_wal.tar.gz = WAL logs")
print("   Together these can restore the database to this exact point in time.")

# Both halves have to exist. base.tar.gz alone cannot be started: it is a
# torn copy of a live data directory, and the WAL is what makes it consistent.
assert 'base.tar.gz' in files, (
    f"pg_basebackup produced no base.tar.gz (got {files}); stderr: {err[-300:]}")
assert 'pg_wal.tar.gz' in files, (
    f"pg_basebackup produced no pg_wal.tar.gz (got {files}) -- without the WAL "
    f"this 'backup' cannot be brought to a consistent state")


## 📚 Backup Verification — The Most Neglected Practice

> **An untested backup is not a backup — it is a hope.**

Many organizations discover their backups are corrupted or incomplete
only when they need to restore them during a real disaster.

### Backup Verification Checklist

1. **Can you restore it?** — Actually restore to a test database
2. **Is the data complete?** — Compare row counts, checksums
3. **How long does restore take?** — This is your actual RTO for backup-based recovery
4. **Can someone else do it?** — Document the procedure, test with different team members
5. **Is the backup accessible?** — Can you reach it during a disaster?

In [ ]:
# =============================================================================
# Demo: Restore and Verify a Backup
# =============================================================================
# We will restore our pg_dump backup into a separate database and verify it.
# Verification is the whole point: a backup you have never restored is a
# hypothesis, not a backup.

print("=" * 65)
print("BACKUP VERIFICATION: Restore + Verify")
print("=" * 65)

# Step 1: Create a test database for restoration
print("\nStep 1: Creating test database for restore...")
docker_exec('bcdr-pg-primary', [
    "psql", "-U", "demo", "-d", "postgres",
    "-c", "DROP DATABASE IF EXISTS bcdr_restore_test"
])
docker_exec('bcdr-pg-primary', [
    "psql", "-U", "demo", "-d", "postgres",
    "-c", "CREATE DATABASE bcdr_restore_test"
])
print("  ✅ Test database created")

# Step 2: Restore the backup
print("\nStep 2: Restoring backup...")
start_time = time.time()
out, err = docker_exec('bcdr-pg-primary', [
    "pg_restore",
    "-U", "demo",
    "-d", "bcdr_restore_test",
    "--no-owner",
    "/tmp/backup.dump"
])
restore_time = time.time() - start_time
assert 'error' not in err.lower(), (
    f"pg_restore reported errors, so this restore is not verified: {err}")
print(f"  ✅ Restore completed in {restore_time:.2f} seconds")

# Step 3: Verify data integrity
print("\nStep 3: Verifying data integrity...")

conn_orig = get_primary_connection()
cur_orig = conn_orig.cursor()

conn_rest = psycopg2.connect(
    host="localhost", port=5432,
    database="bcdr_restore_test", user="demo", password="demo"
)
cur_rest = conn_rest.cursor()


def fingerprint(cur, tbl):
    """Row count PLUS a content hash. Row counts alone are a weak check: a
    restore can produce the right number of rows with the wrong bytes in
    them, and 'the counts matched' is how corrupt backups pass review."""
    cur.execute(
        f"SELECT COUNT(*), "
        f"       md5(COALESCE(string_agg(r, '' ORDER BY r), '')) "
        f"FROM (SELECT t::text AS r FROM {tbl} t) s"
    )
    return cur.fetchone()


tables = ['customers', 'orders', 'order_items', 'payments', 'audit_log']
table_data = []
mismatches = []

for tbl in tables:
    orig_count, orig_md5 = fingerprint(cur_orig, tbl)
    rest_count, rest_md5 = fingerprint(cur_rest, tbl)
    ok = (orig_count == rest_count) and (orig_md5 == rest_md5)
    if not ok:
        mismatches.append(
            f"{tbl}: {orig_count} rows/{orig_md5[:8]} vs "
            f"{rest_count} rows/{rest_md5[:8]}")
    table_data.append([tbl, orig_count, rest_count,
                       orig_md5[:8], rest_md5[:8], "✅" if ok else "❌"])

print(tabulate(table_data,
    headers=["Table", "Original rows", "Restored rows",
             "Orig md5", "Restored md5", "Match"],
    tablefmt="grid"))

conn_orig.close()
conn_rest.close()

# Cleanup
docker_exec('bcdr-pg-primary', [
    "psql", "-U", "demo", "-d", "postgres",
    "-c", "DROP DATABASE IF EXISTS bcdr_restore_test"
])

assert not mismatches, (
    "the restored copy does not match the source: " + "; ".join(mismatches)
    + " -- this backup is NOT verified, whatever the file size says")

print("\n🎉 Row counts AND content hashes match on all five tables.")
print("   This backup is verified — not assumed.")
print(f"\n📊 The restore step took {restore_time:.2f}s. That is ONE COMPONENT of")
print("   a backup-based RTO, not the whole thing: the real number also")
print("   includes noticing the failure, provisioning a host, pulling the")
print("   backup from off-site storage, and replaying WAL to the target time.")


## 📚 Point-in-Time Recovery (PITR)

PITR lets you restore your database to **any specific moment in time**.

```
                  pg_basebackup    accidental    restore
  ─────────────────[BACKUP]─────────[DELETE]──────[HERE]──────
                     ▲                              ▲
                     │     WAL logs fill the gap    │
                     │◄────────────────────────────►│
```

### How It Works

1. Start with a `pg_basebackup` (your base snapshot)
2. PostgreSQL continuously archives WAL files (the change log)
3. To recover: restore the base backup, then replay WAL files up to your target time
4. This gets you to any point between the backup and the disaster

### Requirements
- `archive_mode = on` in PostgreSQL config
- WAL archive storage that survives the disaster
- A known good timestamp to recover to

### And one requirement nobody checks

`archive_mode = on` tells you the archiver is *running*, not that it is
*working*. If the archive destination is unwritable — a permission on a
mounted volume, a full disk, an expired credential — every single attempt
fails, `archive_mode` still reads `on`, and you have no PITR at all. You
find out when you try to use it.

The truth lives in `pg_stat_archiver`: `archived_count` must be climbing and
`failed_count` must be zero. The next cell forces a WAL switch and asserts
exactly that, instead of trusting the setting.


In [ ]:
# =============================================================================
# Demo: Verify WAL Archiving Actually Works
# =============================================================================
# `archive_mode = on` is a claim. pg_stat_archiver is the evidence.

conn = get_primary_connection()
conn.autocommit = True
cur = conn.cursor()

# Check archive settings
cur.execute(
    "SELECT name, setting FROM pg_settings "
    "WHERE name IN ('archive_mode', 'archive_command', 'wal_level') "
    "ORDER BY name"
)
settings = dict(cur.fetchall())

print("=" * 72)
print("WAL ARCHIVE CONFIGURATION")
print("=" * 72)
for name, value in settings.items():
    print(f"  {name}: {value}")

# Force a WAL segment switch so there is definitely something to archive,
# then wait for the archiver to act on it. Without this the counters can be
# legitimately zero on a quiet database and prove nothing either way.
print("\nForcing a WAL switch and waiting for the archiver...")
cur.execute("SELECT pg_switch_wal()")

archived = failed = 0
last_wal = last_time = last_failed = None
deadline = time.time() + 60
while time.time() < deadline:
    cur.execute(
        "SELECT archived_count, failed_count, last_archived_wal, "
        "last_archived_time, last_failed_wal FROM pg_stat_archiver"
    )
    archived, failed, last_wal, last_time, last_failed = cur.fetchone()
    if archived > 0 or failed > 0:
        break
    time.sleep(0.5)

print("\n" + "=" * 72)
print("WAL ARCHIVER STATUS (the part that is usually a surprise)")
print("=" * 72)
print(f"  Archived WAL files: {archived}")
print(f"  Failed archives:    {failed}")
print(f"  Last archived WAL:  {last_wal}")
print(f"  Last archive time:  {last_time}")
print(f"  Last FAILED WAL:    {last_failed}")

# And confirm the bytes really landed on the other side.
out, _ = docker_exec('bcdr-pg-primary',
                     ["ls", "-1", "/var/lib/postgresql/archive"])
files = sorted(l.strip() for l in out.split('\n') if l.strip())
print(f"\n  Files in the archive directory: {len(files)}")
for f in files[:5]:
    print(f"    {f}")

conn.close()

assert archived > 0, (
    f"archive_mode is '{settings.get('archive_mode')}' but NOTHING has ever "
    f"been archived (last failed WAL: {last_failed}). This lab would have no "
    f"point-in-time recovery at all, while every setting looked healthy -- "
    f"which is exactly how it happens in production. Usual cause: the archive "
    f"directory is not writable by the `postgres` user.")
assert failed == 0, (
    f"{failed} archive attempt(s) FAILED (last: {last_failed}). Any gap in the "
    f"WAL archive is a hole in your recovery timeline: you can restore up to "
    f"the gap and no further.")
assert files, "pg_stat_archiver says it archived, but the directory is empty"

print("\n✅ Archiving is not just enabled — it is demonstrably working.")
print("\n💡 WAL archiving + pg_basebackup = point-in-time recovery capability.")
print("   This is how enterprises achieve RPO of minutes or even seconds —")
print("   PROVIDED somebody actually checks these counters. Alert on")
print("   pg_stat_archiver.failed_count; it is a two-line alert that catches")
print("   a class of outage that otherwise stays invisible until the worst day.")


## 📚 The 3-2-1 Backup Rule

A time-tested rule used by IT departments everywhere:

```
   3 copies of your data
   └─ on 2 different types of media
      └─ with 1 copy off-site
```

### Why each number matters

| Number | What | Why |
|--------|------|-----|
| **3 copies** | Your live data + at least 2 backups | Any single copy can fail or be corrupted |
| **2 media types** | e.g., local disk + cloud object storage | Protects against media-specific failures |
| **1 off-site** | In a different building / region / account | Protects against fire, flood, ransomware, rogue admin |

### Modern version: 3-2-1-1-0

Teams add two extras for ransomware defense:
- **1 copy immutable** (write-once, cannot be deleted for N days)
- **0 errors** — every backup is verified by automated restore tests

### What this lab simulates vs. real life

| Layer | This lab | Production BCDR |
|-------|----------|-----------------|
| Live DB | `bcdr-pg-primary` | Primary DB cluster |
| Hot copy | `bcdr-pg-standby` | Replica in another AZ |
| Local backup | `/tmp/backup.dump` in the container | Backup server in the same DC |
| Off-site backup | ⚠️ not simulated | S3 / GCS / Azure Blob in a **different region** |

## 📚 Hands-On: "Replication is NOT a Backup"

This is the single most common BCDR mistake. Let us **prove** it.

### The scenario

1. A developer runs `DELETE FROM orders` without a `WHERE` clause.
2. The delete is **immediately replicated** to the standby.
3. The standby cannot save us — it has the same deleted state.
4. Only a real backup (taken BEFORE the disaster) can bring the data back.

### Bad → Better → Best

| Approach | Outcome |
|----------|---------|
| ❌ **Bad**: "We have a replica, that is our backup" | Data is gone on both primary and replica — unrecoverable |
| ⚠️ **Better**: Nightly `pg_dump` on the same server | Can recover, but up to 24h of data loss, and host-level failure takes the backup too |
| ✅ **Best**: Frequent backups + WAL archiving + off-site (3-2-1) | Recover to any point in time, survive fires and ransomware |

In [ ]:
# =============================================================================
# Demo: Replication is NOT a Backup
# =============================================================================
# We will: take a backup, delete all orders, watch the delete replicate to the
# standby, then restore from the backup.

print('=' * 65)
print("DEMO: 'Replication is NOT a Backup'")
print('=' * 65)

# Step 1: take a fresh backup BEFORE the disaster.
print('\nStep 1: Taking a pg_dump backup (our safety net)...')
docker_exec('bcdr-pg-primary', [
    'pg_dump', '-U', 'demo', '-d', 'bcdr_demo',
    '--format=custom', '--file=/tmp/pre_disaster.dump',
])
print('  ✅ Backup saved to /tmp/pre_disaster.dump')

# Open connections. Use autocommit from the start: psycopg2 will not let you
# change session state once a query has opened a transaction.
pconn = get_primary_connection()
pconn.autocommit = True
pcur = pconn.cursor()

sconn = psycopg2.connect(host='localhost', port=55433,
                         database='bcdr_demo', user='demo', password='demo')
sconn.autocommit = True
scur = sconn.cursor()

# The entire point of this demo is that 55433 is a LIVE replica. If replication
# were broken, the DELETE below simply would not show up there and the demo
# would happily "prove" the opposite of its own lesson. Check before claiming.
scur.execute('SELECT pg_is_in_recovery()')
assert scur.fetchone()[0], (
    "port 55433 is not a standby in recovery -- run the failback cell at the "
    "top of this notebook. Without live replication this demo proves nothing.")

# Step 2: record order counts on primary + standby
pcur.execute('SELECT COUNT(*) FROM orders'); p_before = pcur.fetchone()[0]
scur.execute('SELECT COUNT(*) FROM orders'); s_before = scur.fetchone()[0]
print(f'\nStep 2: Before disaster → primary: {p_before} orders, standby: {s_before} orders')
assert p_before > 0, 'there is nothing to delete -- the primary has no orders'
assert s_before == p_before, (
    f'the standby is already behind ({s_before} vs {p_before} orders) before '
    f'the demo even starts')

# Step 3: the 'disaster' — a developer deletes everything.
print('\nStep 3: 💥 Running "DELETE FROM orders" on PRIMARY...')
# Start the clock BEFORE the statement, so what we measure is "how long from
# issuing the mistake to the mistake being live on the replica" -- not the
# time from noticing it, which would flatter the number to ~0.
delete_start = time.time()
pcur.execute('DELETE FROM payments')     # child rows first (FK)
pcur.execute('DELETE FROM order_items')
pcur.execute('DELETE FROM orders')
pcur.execute('SELECT COUNT(*) FROM orders'); p_after = pcur.fetchone()[0]

# Poll the standby rather than sleeping: we want to WATCH the delete arrive,
# and a fixed sleep would hide how fast the mistake propagates.
delete_seen_at = None
deadline = time.time() + 15
s_after = s_before
while time.time() < deadline:
    scur.execute('SELECT COUNT(*) FROM orders')
    s_after = scur.fetchone()[0]
    if s_after == p_after:
        delete_seen_at = time.time() - delete_start
        break
    time.sleep(0.05)

print(f'  After DELETE → primary: {p_after} orders, standby: {s_after} orders')
assert p_after == 0, f'the DELETE did not empty the primary ({p_after} left)'
assert s_after == 0, (
    f'the DELETE never reached the standby ({s_after} orders still there) -- '
    f'replication is broken, so this demo is not showing what it claims')
print(f'  ❌ The standby was NOT a safety net: all {s_before} orders vanished')
print(f'     there too, {delete_seen_at*1000:.1f} ms after the statement ran.')
print('     Replication is a copying machine — it copies your mistakes just')
print('     as faithfully as your good writes.')

# Step 4: restore from the backup we took in step 1.
# --data-only restores rows into the existing schema, and we restore ONLY the
# three tables we deleted. Restoring every table would try to re-insert the
# audit_log rows that are still live and fail on the primary key.
print('\nStep 4: Restoring from pre-disaster backup...')
out, err = docker_exec('bcdr-pg-primary', [
    'pg_restore', '-U', 'demo', '-d', 'bcdr_demo',
    '--data-only', '--disable-triggers',
    '-t', 'orders', '-t', 'order_items', '-t', 'payments',
    '/tmp/pre_disaster.dump',
])
pcur.execute('SELECT COUNT(*) FROM orders'); p_recovered = pcur.fetchone()[0]
print(f'  ✅ Primary after restore: {p_recovered} orders')
assert p_recovered == p_before, (
    f'the restore brought back {p_recovered} of {p_before} orders -- an '
    f'unverified restore is exactly the GitLab failure mode. pg_restore said: '
    f'{err[-300:]}')

# ...and the recovery has to replicate back out to the standby too, or you
# have repaired one node and left the other one empty.
s_recovered = 0
deadline = time.time() + 15
while time.time() < deadline:
    scur.execute('SELECT COUNT(*) FROM orders'); s_recovered = scur.fetchone()[0]
    if s_recovered == p_recovered:
        break
    time.sleep(0.05)
print(f'  ✅ Standby after restore:  {s_recovered} orders')
assert s_recovered == p_recovered, (
    f'the restored rows did not replicate to the standby '
    f'({s_recovered} vs {p_recovered})')

print('\n💡 Takeaways:')
print('   • Replication protects against HARDWARE failure, not LOGICAL errors.')
print('   • Always keep real backups — and keep some OFF-SITE (3-2-1 rule).')
print('   • For near-zero RPO on logical errors, enable point-in-time recovery.')
print(f'   • Note the RPO of this recovery: everything written between the')
print(f'     backup in step 1 and the DELETE in step 3 would have been lost.')
print(f'     Nightly backups make that window 24 hours wide.')

pconn.close(); sconn.close()


## 📝 Summary

### What You Learned

1. **Replication is NOT backup** — Deletes and corruption replicate too!
2. **Full/Incremental/Differential** — Trade-offs between backup speed, size, and restore speed.
3. **pg_dump** — Logical backup (SQL). Good for small DBs, selective restore, cross-version.
4. **pg_basebackup** — Physical backup (binary). Good for large DBs and PITR.
5. **Backup verification** — Always restore and verify. Measure your actual restore time.
6. **PITR** — Base backup + WAL archive = restore to any point in time.

### Key Takeaway

> **An untested backup is not a backup. Schedule regular restore tests.**

### Next Notebook

In **Notebook 4**, we run a full disaster recovery drill — simulating
a primary failure and measuring our actual RTO.